In [ ]:
import sys
PATH_TO_TOP = "/genra"
if PATH_TO_TOP not in sys.path:
    sys.path.append(PATH_TO_TOP)
import os
os.chdir(PATH_TO_TOP)
from IPython.display import JSON

In [ ]:
from genraweb.resources import DB

In [ ]:
# DB.g2025_invitrodb_assay_rslt.drop()
# DB.g2025_toxcast_assays.drop()

#### Checking DB document values

In [ ]:
JSON(DB.mm_txct4.find_one({}, {"_id": 0}))

In [ ]:
DB.mm_txct4.count_documents({})

In [ ]:
DB.mm_txct.count_documents({})

In [ ]:
JSON(DB.toxcast_fp.find_one({"dsstox_sid":"DTXSID0020070"}, {"_id":0}))

In [ ]:
JSON(DB.g2025_invitrodb_assay_rslt.find_one({"dsstox_sid":"DTXSID0020020", "modl_tp":}, {"_id":0}))

### Descrepencies from api (get data by dtxsid)
 * dsstox_sid is dtxsid
 * name is chnm
 * modl_* fields are found in mc5param["*"]
 * **modl_acc not found**
 * other snake case (e.g. resp_max) are camel case in api

In [ ]:
JSON(DB.toxcast_assays.find_one({}, {"_id":0}))

### Descrepencies from api (get all assays)
https://api-ccte.epa.gov/bioactivity/assay/ maps to toxcast_assays collection, however all camel case has to be converted to snake case
other descrepencies:
 * several fields are in a sub-dictionary "gene" in the api: geneId, geneName, geneSymbol, organismId, trackStatus, entrezGeneId, officialSymbol, officialFullName, uniprotAccessionNumber
 * missing fields: 'signal_direction_type', 'analysis_direction', 'fit_all', 'taxon_name', 'common_name'

### Get assays for each dtxsid

In [ ]:
from requests import get
from getpass import getpass

In [ ]:
api_key = getpass()

In [ ]:
dtxsid = 'DTXSID0021125'

In [ ]:
headers = {
    'accept': 'application/hal+json',
    'x-api-key' : api_key,
}
url = f'https://api-ccte.epa.gov/bioactivity/data/search/by-dtxsid/{dtxsid}'
resp = get(url, headers=headers)
print(resp.ok)
print(resp.status_code)

In [ ]:
assays = resp.json()

In [ ]:
# parse assays example
mongo_assays = []
for assay in assays:
    mongo_assays.append({
        "dsstox_sid": assay["dtxsid"],
        "name": assay["chnm"],
        "aeid": assay["aeid"],
        "modl": assay["modl"],
        "hitc": assay["hitc"],
        "fitc": assay["fitc"],
        "coff": assay["coff"],
        "modl_er": assay["mc5Param"].get("er"),
        "modl_rmse": assay["mc5Param"].get("rmse"),
        # "modl_acc": not found,
        "modl_ac10": assay["mc5Param"].get("ac10"),
        "bmad": assay["bmad"],
        "resp_max": assay["respMax"],
        "resp_min": assay["respMin"],
        "max_mean": assay["maxMean"],
        "max_med": assay["maxMed"],
        "max_mean_conc": assay["maxMeanConc"],
        "max_med_conc": assay["maxMedConc"],
        "logc_max": assay["logcMax"],
        "logc_min": assay["logcMin"],
        "nconc": assay["nconc"],
        "npts": assay["npts"],
        "nrep": assay["nrep"],
    })
len(mongo_assays)

In [ ]:
JSON(mongo_assays[0])

### Get assays for all DTXSIDs

In [ ]:
resp = get("https://api-ccte.epa.gov/chemical/list/chemicals/search/by-listname/ToxCast_invitroDB_v4_1", headers=headers)
resp.ok

In [ ]:
chem_list = resp.json()
assert len(chem_list) == 9559
# chem_list = list(set(chem_list) - set(os.listdir("/genra/misc/tickets/GEN-1310-invitrodb-4/cache")))
print(len(chem_list))

In [ ]:
!pip install aiohttp
import aiohttp
import asyncio
import json

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# test async version with one id
ex_dtxsid = 'DTXSID2045238'
url = f'https://api-ccte.epa.gov/bioactivity/data/search/by-dtxsid/{ex_dtxsid}'
async with aiohttp.ClientSession() as session:
    resp = await session.get(url, headers=headers)
    print(resp.status)
    print(await resp.json())

In [ ]:
def format_assays(assays):
    return_assays = []
    for assay in assays:
        return_assays.append({
            "dsstox_sid": assay["dtxsid"],
            "name": assay["chnm"],
            "aeid": assay["aeid"],
            "modl": assay["modl"],
            "hitc": assay["hitc"],
            "fitc": assay["fitc"],
            "coff": assay["coff"],
            "modl_er": assay["mc5Param"].get("er"),
            "modl_ga": assay["mc5Param"].get("ga"),
            "modl_la": assay["mc5Param"].get("la"),
            "modl_rmse": assay["mc5Param"].get("rmse"),
            "modl_acc": assay["mc5Param"].get("acc"),
            "modl_ac10": assay["mc5Param"].get("ac10"),
            "modl_tp": assay["mc5Param"].get("tp"),
            "bmad": assay["bmad"],
            "resp_max": assay["respMax"],
            "resp_min": assay["respMin"],
            "max_mean": assay["maxMean"],
            "max_med": assay["maxMed"],
            "max_mean_conc": assay["maxMeanConc"],
            "max_med_conc": assay["maxMedConc"],
            "logc_max": assay["logcMax"],
            "logc_min": assay["logcMin"],
            "nconc": assay["nconc"],
            "npts": assay["npts"],
            "nrep": assay["nrep"],
        })
    return return_assays

In [ ]:
import json
from pathlib import Path
async def get_assays(session, dtxsid):
    url = f'https://api-ccte.epa.gov/bioactivity/data/search/by-dtxsid/{dtxsid}'
    cache_path = Path("/genra/misc/tickets/GEN-1310-invitrodb-4/cache") / dtxsid
    # print(f"Checking {cache_path}")
    if cache_path.exists():
        with cache_path.open() as cached:
            return json.load(cached)
    async with session.get(url, headers=headers) as resp:
        try:
            data = await resp.content.read()
            if not resp.ok or resp.status != 200:  # Getting 202 which is OK but not OK
                print(f"bad status code {dtxsid}")
                print(data[:80])
                missing.append(dtxsid)
                return []
            assays = json.loads(data)
            with cache_path.open("w") as cached:
                json.dump(assays, cached)
            return assays
        except json.JSONDecodeError:
            print(f"unable to parse {dtxsid}")
            print(data[:80])
            print(response.status)
            missing.append(dtxsid)
            return []

In [ ]:
# async get all assays to make it faster
import itertools
async def get_all_assays(chem_list):
    conn = aiohttp.TCPConnector(limit_per_host=20)
    async with aiohttp.ClientSession(connector=conn) as session:
        tasks = []
        for dtxsid in chem_list:
            tasks.append(asyncio.ensure_future(get_assays(session, dtxsid)))
        async_assays = await asyncio.gather(*tasks)
    return list(itertools.chain.from_iterable(async_assays))


In [ ]:
%%time
missing = []
all_assays = asyncio.run(get_all_assays(chem_list))
print(f"assays found:{len(all_assays)}\nDTXSIDs with errors: {len(missing)}")

In [ ]:
print(len(missing))

In [ ]:
%%time
# get data from dtxsids that failed the first time (not sure why)
new_chem_list=missing.copy()
missing = []
all_assays += asyncio.run(get_all_assays(new_chem_list))
print(f"assays found:{len(all_assays)}\nDTXSIDs with errors: {len(missing)}")

In [ ]:
mongo_assays = format_assays(all_assays)

In [ ]:
print(len(mongo_assays))

In [ ]:
_ = DB.g2025_invitrodb_assay_rslt.insert_many(mongo_assays)
# Don't try and print this result, runs out of browser mem.
del _

In [ ]:
DB.g2025_invitrodb_assay_rslt.count_documents({})

In [ ]:
DB.toxcast4_fp.count_documents({})

In [ ]:
from genraweb.lib.fp.fpclass import FPGen
print(list(FPGen.FPClass))
tc4 = FPGen.FPClass['bio_txct4'](DB, "toxcast4_fp")
tc4.generate_fps(["DTXSID7020182"])

### Get toxcast_assays data

In [ ]:
len(DB.toxcast_assays.distinct('aeid'))

In [ ]:
url = "https://api-ccte.epa.gov/bioactivity/assay/"
resp = get(url, headers=headers)
toxcast_assays_json = resp.json()
len(toxcast_assays_json)

In [ ]:
# format data for genra
formatted_toxcast_assays = []

for assay in toxcast_assays_json:
    gene_data = assay["gene"]
    if gene_data is None:
        gene_data = dict()
    formatted_toxcast_assays.append(dict(
        aeid=assay["aeid"],
        assay_component_endpoint_name=assay["assayComponentEndpointName"],
        export_ready=assay["exportReady"],
        internal_ready=assay["internalReady"],
        assay_component_endpoint_desc=assay["assayComponentEndpointDesc"],
        assay_function_type=assay["assayFunctionType"],
        normalized_data_type=assay["normalizedDataType"],
        burst_assay=assay["burstAssay"],
        key_positive_control=assay["keyPositiveControl"],
        signal_direction=assay["signalDirection"],
        intended_target_type=assay["intendedTargetType"],
        intended_target_type_sub=assay["intendedTargetTypeSub"],
        intended_target_family=assay["intendedTargetFamily"],
        intended_target_family_sub=assay["intendedTargetFamilySub"],
        cell_viability_assay=assay["cellViabilityAssay"],
        data_usability=assay["dataUsability"],
        acid=assay["acid"],
        assay_component_name=assay["assayComponentName"],
        assay_component_desc=assay["assayComponentDesc"],
        assay_component_target_desc=assay["assayComponentTargetDesc"],
        parameter_readout_type=assay["parameterReadoutType"],
        assay_design_type=assay["assayDesignType"],
        assay_design_type_sub=assay["assayDesignTypeSub"],
        biological_process_target=assay["biologicalProcessTarget"],
        detection_technology_type=assay["detectionTechnologyType"],
        detection_technology_type_sub=assay["detectionTechnologyTypeSub"],
        detection_technology=assay["detectionTechnology"],
        key_assay_reagent_type=assay["keyAssayReagentType"],
        key_assay_reagent=assay["keyAssayReagent"],
        technological_target_type=assay["technologicalTargetType"],
        technological_target_type_sub=assay["technologicalTargetTypeSub"],
        aid=assay["aid"],
        assay_name=assay["assayName"],
        assay_desc=assay["assayDesc"],
        timepoint_hr=assay["timepointHr"],
        organism_id=assay["organismId"],
        organism=assay["organism"],
        taxon_name=assay["organism"],
        common_name=assay["organism"],
        tissue=assay["tissue"],
        cell_format=assay["cellFormat"],
        cell_free_component_source=assay["cellFreeComponentSource"],
        cell_short_name=assay["cellShortName"],
        cell_growth_mode=assay["cellGrowthMode"],
        assay_footprint=assay["assayFootprint"],
        assay_format_type=assay["assayFormatType"],
        assay_format_type_sub=assay["assayFormatTypeSub"],
        content_readout_type=assay["contentReadoutType"],
        dilution_solvent=assay["dilutionSolvent"],
        dilution_solvent_percent_max=assay["dilutionSolventPercentMax"],
        asid=assay["asid"],
        assay_source_name=assay["assaySourceName"],
        assay_source_long_name=assay["assaySourceLongName"],
        assay_source_desc=assay["assaySourceDesc"],
        gene_id=gene_data.get("geneId"),
        gene_name=gene_data.get("geneName"),
        description=gene_data.get("description"),
        gene_symbol=gene_data.get("geneSymbol"),
        track_status=gene_data.get("trackStatus"),
        entrez_gene_id=gene_data.get("entrezGeneId"),
        official_symbol=gene_data.get("officialSymbol"),
        official_full_name=gene_data.get("officialFullName"),
        uniprot_accession_number=gene_data.get("uniprotAccessionNumber"),
    ))

In [ ]:
len(formatted_toxcast_assays)

In [ ]:
_ = DB.g2025_toxcast_assays.insert_many(formatted_toxcast_assays)

### Compare new toxcast vs genra

In [ ]:
from deepdiff import DeepDiff

In [ ]:
all_new_keys = set()
for doc in DB.g2025_toxcast_assays.find({},{"_id": 0}):
    all_new_keys.update(doc.keys())

all_keys = set()
for doc in DB.toxcast_assays.find({},{"_id": 0}):
    all_keys.update(doc.keys())

In [ ]:
from deepdiff import DeepDiff
JSON(DeepDiff(all_keys, all_new_keys))

We lose `signal_direction_type, fit_all, and analysis_direction`, which are no longer necessary per Jason's email.